In [ ]:
import os
import re
import pickle
import matplotlib.pyplot as plt
import numpy as np
import math

In [ ]:
rename_dict = {
    'M1': 'casesonly_static',
    'M2': 'casesonly_mobs',
    'M3': 'casesonly_sem',
    'M4': 'cases_tweets_static',
    'M5': 'cases_tweets_mobs',
    'M6': 'cases_tweets_sem',
    'M7': 'cases_randts_static',
    'M8': 'cases_randts_mobs',
    'M9': 'cases_randts_sem',
    'M10': 'casesonly_static_randadj',
    'M11': 'casesonly_dynamic_randadj',
    'M12': 'cases_tweets_static_randadj',
    'M13': 'cases_tweets_dynamic_randadj',
}

In [ ]:
def extract_model_ids_from_titles(titles):
    """
    Extract model IDs (e.g., 'M10', 'M2', ...) from the beginning of each title string.

    Args:
        titles (list of str): Titles like "M1: Something".

    Returns:
        list of str: Extracted model IDs.
    """
    model_ids = []
    for title in titles:
        match = re.match(r'^(M\d+):', title.strip())
        if match:
            model_ids.append(match.group(1))
        else:
            raise ValueError(f"Could not extract model ID from title: {title}")
    return model_ids

In [ ]:
def load_model_results(model_ids, model_dir, titles=None, return_dict=False, model_specifics=''):
    """
    Load multiple model results from a given directory.

    Args:
        model_ids (list of str): e.g., ['M1', 'M2', ...].
        model_dir (str): Directory containing result pickle files.
        titles (list of str, optional): Descriptive titles for each model.
        return_dict (bool): If True, return dict keyed by lowercase model IDs.

    Returns:
        list: Loaded results in model_ids order.
        dict (optional): Dict with structure { 'm1': {'result': ..., 'title': ...}, ... }
    """
    all_results_list = []
    model_results_dict = {}

    for idx, model_id in enumerate(model_ids):
        filename = f"training_results_{model_id}{model_specifics}.pkl"
        results_path = os.path.join(model_dir, filename)

        with open(results_path, 'rb') as f:
            result = pickle.load(f)
            all_results_list.append(result)

            if titles:
                model_results_dict[model_id.lower()] = {
                    'result': result,
                    'title': titles[idx]
                }
            else:
                model_results_dict[model_id.lower()] = {
                    'result': result,
                    'title': model_id
                }

    if return_dict:
        return all_results_list, model_results_dict
    else:
        return all_results_list


    """
    Plot training and test loss (with ±1σ) for multiple models in a grid.

    Args:
        results_list (list): List of model result objects (as from `load_model_results`).
        titles (list): Titles for each subplot (e.g., model descriptions).
        n_rows (int): Grid rows.
        n_cols (int): Grid columns.
        suptitle (str): Overall title.
        ylim_max (float): Maximum y-limit for loss plots.

    Returns:
        None
    """
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4.5 * n_rows))
    axes = axes.flatten()

    # Compute global y-limits
    all_losses = []
    for results in results_list:
        for run in results:
            all_losses.extend(run['train_losses'])
            all_losses.extend(run['test_losses'])

    ymin = min(all_losses)
    ymax = ylim_max
    padding = (ymax - ymin) * 0.05
    ymin -= padding

    for i, results in enumerate(results_list):
        train_arr = np.array([r['train_losses'] for r in results])
        test_arr  = np.array([r['test_losses'] for r in results])

        mean_train = train_arr.mean(axis=0)
        std_train  = train_arr.std(axis=0)
        mean_test  = test_arr.mean(axis=0)
        std_test   = test_arr.std(axis=0)

        epochs = np.arange(len(mean_train))
        ax = axes[i]

        ax.plot(epochs, mean_train, label='Mean Train Loss')
        ax.fill_between(epochs,
                        mean_train - std_train,
                        mean_train + std_train,
                        alpha=0.2,
                        label='Train ±1σ')

        ax.plot(epochs, mean_test, label='Mean Test Loss')
        ax.fill_between(epochs,
                        mean_test - std_test,
                        mean_test + std_test,
                        alpha=0.2,
                        label='Test ±1σ')

        ax.set_title(titles[i])
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_ylim(ymin, ymax)
        ax.grid(True)
        ax.legend(loc='upper right')

    if suptitle:
        plt.suptitle(suptitle, fontsize=18)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

In [ ]:
def plot_model_losses(results_list, titles, suptitle=None, ylim_max=1):
    """
    Plot training and test loss (with ±1σ) for multiple models in a grid,
    and print final train/test losses from all runs per model.

    Args:
        results_list (list): List of model result objects (as from `load_model_results`).
        titles (list): Titles for each subplot (e.g., model descriptions).
        suptitle (str): Overall title.
        ylim_max (float): Maximum y-limit for loss plots.

    Returns:
        None
    """
    n_models = len(results_list)

    # Dynamic layout
    if n_models <= 1:
        n_rows, n_cols = 1, 1
    elif n_models == 2:
        n_rows, n_cols = 1, 2
    elif n_models <= 4:
        n_rows, n_cols = 2, 2
    elif n_models <= 6:
        n_rows, n_cols = 2, 3
    elif n_models <= 9:
        n_rows, n_cols = 3, 3
    else:
        n_cols = min(4, n_models)
        n_rows = math.ceil(n_models / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4.5 * n_rows))
    axes = np.array(axes).flatten()

    # Compute global y-limits
    all_losses = []
    for results in results_list:
        for run in results:
            all_losses.extend(run['train_metrics']['mse'])


    ymin = min(all_losses)
    ymax = ylim_max
    padding = (ymax - ymin) * 0.05
    ymin -= padding

    for i, results in enumerate(results_list):
        train_arr = np.array([r['train_metrics']['mse'] for r in results])
        test_arr  = np.array([r['valid_metrics']['mse'] for r in results])

        mean_train = train_arr.mean(axis=0)
        std_train  = train_arr.std(axis=0)
        mean_test  = test_arr.mean(axis=0)
        std_test   = test_arr.std(axis=0)

        epochs = np.arange(len(mean_train))
        ax = axes[i]

        ax.plot(epochs, mean_train, label='Mean Train Loss')
        ax.fill_between(epochs, mean_train - std_train, mean_train + std_train, alpha=0.2, label='Train ±1σ')

        ax.plot(epochs, mean_test, label='Mean Validation Loss')
        ax.fill_between(epochs, mean_test - std_test, mean_test + std_test, alpha=0.2, label='Validation ±1σ')

        ax.set_title(titles[i])
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_ylim(ymin, ymax)
        ax.grid(True)
        ax.legend(loc='upper right')

    # Hide unused axes
    for j in range(len(results_list), len(axes)):
        fig.delaxes(axes[j])

    if suptitle:
        plt.suptitle(suptitle, fontsize=18)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    # Print averaged final losses (last epoch) per model
    print("\nFinal Losses per Model (average of last epoch across runs):\n")
    for row in range(n_rows):
        row_output = ""
        for col in range(n_cols):
            idx = row * n_cols + col
            if idx >= n_models:
                continue
            title = titles[idx]
            results = results_list[idx]

            final_train_losses = [run['train_metrics']['mse'] [-1] for run in results]
            final_test_losses = [run['valid_metrics']['mse'] [-1] for run in results]

            avg_train = np.mean(final_train_losses)
            avg_test = np.mean(final_test_losses)

            row_output += f"{title}:\n  Avg Train: {avg_train:.4f}, Avg Validation: {avg_test:.4f}\n"
        print(row_output)

In [ ]:
model_dir = r"\3_Data\model_checkpoints (11)\fold_153"
model_specifics= '_adamw_0.0001_rmse_zscore_RescalingOfAdjs_NewTsButterNoLog_16folds' #'_sgd_0.0001_mse_zscore_sgd_NoRescalingOfAdjs'

model_ids = ['M1', 'M2', 'M3', 'M4', 'M5', 'M6']
titles = [
    "M1: Cases + Spatial Distance",
    "M2: Cases + User Mobility",
    "M3: Cases + Semantic Similarity",
    "M4: Cases & Geo-Social Media Posts + Spatial Distance",
    "M5: Cases & Geo-Social Media Posts + User Mobility",
    "M6: Cases & Geo-Social Media Posts + Semantic Similarity"
]


# 1. Load results  _adamw_0.0001_mse_boxcox_adamw_NoRescalingOfAdjs
results = load_model_results(model_ids, model_dir, titles, model_specifics=model_specifics)

# 2. Plot results
plot_model_losses(results, titles, suptitle='Average Train and Validation Loss Across 6 Models')

#
# Only cases: random edges vs real edges
#
titles = [
    # first row
    "M10: Cases + Random Static Edges", "M11: Cases + Random Dynamic Edges", "M11: Cases + Random Dynamic Edges",#"M13: Cases + Geo-Social Media Posts + Random Dynamic Edges",
    # second row
    "M1: Cases + Spatial Distance", "M2: Cases + User Mobility", "M3: Cases + Semantic Similarity",
]
model_ids = extract_model_ids_from_titles(titles)
print(model_ids)

# 1. Load results
results = load_model_results(model_ids, model_dir, titles, model_specifics=model_specifics)

# 2. Plot results
plot_model_losses(results, titles, suptitle='Average Train and Test Loss Across 6 Models')

